# Clasificación de Deserción Estudiantil con SVM: Flujo Reproducible Sin Fuga de Datos

**Proyecto**: Machine Learning II - Universidad 2026-II  
**Autor**: Arnold Santiago Torres  
**Dataset**: `estudiantes_sinteticos_11500.csv` (11,500 registros, 21 variables socioeconómicas y académicas)  

---  
### Sección 7.2: Flujo Reproducible Sin Fuga de Datos (Data Leakage Prevention)
Se integra el preprocesamiento (`ColumnTransformer`: codificación de categóricas + escalamiento numérico) y el estimador (`SVC`) en un **Pipeline** reproducible de Scikit-Learn.  
Esto garantiza que **todas las transformaciones se ajusten (`fit`) ÚNICAMENTE con los datos de entrenamiento de cada partición (`X_train`)**, evitando estrictamente la contaminación o fuga de información hacia el conjunto de prueba (`X_test`).
---

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed, dump

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, MaxAbsScaler,
    QuantileTransformer, PowerTransformer, OrdinalEncoder, OneHotEncoder, FunctionTransformer
)
from scipy.stats import ttest_ind

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['font.sans-serif'] = 'Arial'

N_JOBS = 2  # Usar 2 núcleos del procesador
print(f"Configuración de ejecuciones con Pipelines usando {N_JOBS} núcleos de la CPU.")

## 1. Carga de Datos y EDA Previo

In [ ]:
data_path = 'estudiantes_sinteticos_11500.csv'
df = pd.read_csv(data_path)

riesgo = (
    (df['promedio_academico'] < 3.0).astype(int) * 2.5 +
    (df['materias_reprobadas'] >= 2).astype(int) * 2.5 +
    (df['puntaje_icfes_percentil'] < 40).astype(int) * 1.0 +
    (df['trabaja_mientras_estudia'] == 1).astype(int) * 1.0 +
    (df['beneficiario_beca'] == 0).astype(int) * 0.5 +
    (df['estrato_socioeconomico'] <= 2).astype(int) * 0.5
)
df['desercion'] = (riesgo >= 3.5).astype(int)

cat_cols = ['sexo', 'estado_civil', 'zona_residencia', 'departamento',
            'nivel_educativo_padre', 'nivel_educativo_madre', 'sector_ies',
            'nivel_formacion', 'metodologia', 'area_conocimiento']
num_cols = ['edad_ingreso', 'promedio_academico', 'materias_reprobadas',
            'distancia_hogar_ies_km', 'puntaje_icfes_percentil', 'semestre_cursado',
            'estrato_socioeconomico', 'anio_registro', 'trabaja_mientras_estudia',
            'beneficiario_icetex', 'beneficiario_beca']

X_raw = df[num_cols + cat_cols]
y = df['desercion'].values

print(f"Dimensiones: {df.shape[0]} estudiantes y {df.shape[1]} variables.")
print(f"Valores nulos: {df.isnull().sum().sum()}")
print("\nDistribución del Target Binario (1 = Deserción, 0 = No Deserción):")
print(pd.Series(y).value_counts(normalize=True) * 100)

## 7.2. Flujo Reproducible Sin Fuga de Datos (`Pipeline` + `ColumnTransformer`)

### Justificación Metodológica de Prevención de Fuga de Datos:
1. **Aislamiento Estricto de Particiones**: Si se aplicara la codificación One-Hot o el cálculo del promedio/desviación estándar sobre toda la matriz $X$ antes de hacer `train_test_split`, la información de los estudiantes del conjunto de prueba se filtraría hacia las estadísticas del escalador (**Data Leakage**).
2. **Etapas del Pipeline**:
   - **Partición**: Separación cruda en `X_train`, `X_test`, `y_train`, `y_test`.
   - **ColumnTransformer**: Ajusta el codificador y el escalador **estrictamente en `X_train`**.
   - **SVC Estimador**: Entrena el hiperplano con las características transformadas únicamente de `X_train`.
   - **Inferencia**: Aplica `.predict(X_test)` pasando `X_test` crudo por las transformaciones aprendidas previamente.

In [ ]:
normalization_methods = {
    'Standard Scaler': StandardScaler(),
    'Min Max Scaler': MinMaxScaler(),
    'Quantile Transformer': QuantileTransformer(n_quantiles=50, random_state=42, output_distribution='uniform'),
    'Max Absolute Scaler': MaxAbsScaler(),
    'Power Transformer': PowerTransformer(method='yeo-johnson'),
    'No Normalization (Baseline)': FunctionTransformer(validate=False)
}

N_TRIALS = 10

def evaluate_single_trial(trial_idx):
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y, test_size=0.2, random_state=42 + trial_idx, stratify=y
    )
    trial_scores = {}
    for method_name, num_scaler in normalization_methods.items():
        try:
            num_scaler_inst = type(num_scaler)(**num_scaler.get_params()) if hasattr(num_scaler, 'get_params') else num_scaler
            preprocessor = ColumnTransformer(
                transformers=[
                    ('num', num_scaler_inst, num_cols),
                    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols)
                ]
            )
            pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', SVC(kernel='rbf', C=1.0, cache_size=1000, random_state=42))
            ])
            pipeline.fit(X_train, y_train)
            preds = pipeline.predict(X_test)
            trial_scores[method_name] = f1_score(y_test, preds)
        except Exception:
            trial_scores[method_name] = np.nan
    return trial_scores

print(f"Ejecutando {N_TRIALS} iteraciones con Pipelines sin fuga de datos (n_jobs={N_JOBS})...")
parallel_results = Parallel(n_jobs=N_JOBS, prefer="threads")(delayed(evaluate_single_trial)(t) for t in range(N_TRIALS))

results = {method: [] for method in normalization_methods.keys()}
for res in parallel_results:
    for method_name, score in res.items():
        results[method_name].append(score)

baseline_scores = results['No Normalization (Baseline)']
summary_rows = []
for method_name, scores in results.items():
    mean_f1 = np.mean(scores)
    std_f1 = np.std(scores)
    p_val = ttest_ind(scores, baseline_scores, equal_var=False)[1] if method_name != 'No Normalization (Baseline)' else 1.0
    summary_rows.append({
        'Normalization Method': method_name,
        'Average F1 Score': mean_f1,
        'Std Dev': std_f1,
        'P-value vs Baseline': p_val
    })

summary_df = pd.DataFrame(summary_rows).sort_values(by='Average F1 Score', ascending=False)
display(summary_df)

In [ ]:
plt.figure(figsize=(12, 6))
colors_bar = ['#d95f02' if 'Baseline' in name else '#1b9e77' for name in summary_df['Normalization Method']]
barplot = sns.barplot(data=summary_df, x='Average F1 Score', y='Normalization Method', palette=colors_bar)
plt.title('Evaluación de Pipelines SVM sin Fuga de Datos: Escaladores vs Baseline', fontsize=14, fontweight='bold')
plt.xlabel('F1-Score Promedio (10 iteraciones)', fontsize=12)
plt.xlim(0.4, 1.0)
plt.show()

## 3. Importancia de Variables en Pipeline SVM (`n_jobs=2`)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42, stratify=y)

best_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

best_pipeline = Pipeline(steps=[
    ('preprocessor', best_preprocessor),
    ('classifier', SVC(kernel='rbf', C=1.0, probability=True, cache_size=1000, random_state=42))
])

best_pipeline.fit(X_train, y_train)

# Guardar el Pipeline entrenado e inmutable para inferencia reproducible
dump(best_pipeline, 'svm_pipeline_reproducible.joblib')
print("Pipeline guardado exitosamente en 'svm_pipeline_reproducible.joblib'")

perm_importance = permutation_importance(
    best_pipeline, X_test, y_test, n_repeats=5, random_state=42, n_jobs=N_JOBS
)

sorted_idx = perm_importance.importances_mean.argsort()[::-1]
imp_df = pd.DataFrame({
    'Feature': X_raw.columns[sorted_idx],
    'Importance_Mean': perm_importance.importances_mean[sorted_idx],
    'Importance_Std': perm_importance.importances_std[sorted_idx]
})

plt.figure(figsize=(10, 7))
top_15_imp = imp_df.head(15)
plt.barh(top_15_imp['Feature'][::-1], top_15_imp['Importance_Mean'][::-1], color='#2b5c8f', xerr=top_15_imp['Importance_Std'][::-1])
plt.title('Top 15 Variables Más Relevantes en el Pipeline SVM', fontsize=14, fontweight='bold')
plt.xlabel('Disminución Promedio en F1-Score al Permutar Variable', fontsize=12)
plt.tight_layout()
plt.show()
display(imp_df.head(10))